In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# pip installs

In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes

In [ ]:
import os
from huggingface_hub import snapshot_download

# This is a safe, visible directory
local_model_path = "/kaggle/working/llama-3-1-clean"

snapshot_download(
    repo_id="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    local_dir=local_model_path,
    local_dir_use_symlinks=False, # FORCES real files, no broken pointers
    resume_download=True
)

# Sentence Transformer


In [ ]:
# from sentence_transformers import SentenceTransformer
# from sentence_transformers import util
# sim_model = SentenceTransformer('all-MiniLM-L6-v2')

# sentence1 = ['Self does not exist and is an appearance in consciousness', 'I think therefore, I am']
# sentence2 = ['Self does not exist and is an appearance in consciousness', 'I is an emergent property born out of survivability rather than personal will. ']

# embeddings1 = sim_model.encode(sentence1)
# embeddings2 = sim_model.encode(sentence2)

# cs1 = util.cos_sim(embeddings1[0], embeddings1[1])
# cs2 = util.cos_sim(embeddings2[0], embeddings2[1])
# print(cs1)
# print(cs2)

# LOADING LLAMA 8.1

In [ ]:
from unsloth import FastLanguageModel
import torch
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = "/kaggle/working/llama-3-1-clean", # No more cache!
    max_seq_length = 2048,
    dtype        = None,
    load_in_4bit = True,
)

In [ ]:
# Apply LoRA adapters

adapter_path = '/kaggle/input/datasets/mehulkumar99/llama3-1-8b-checkpoints/outputs/checkpoint-50'
model = PeftModel.from_pretrained(model, adapter_path)

print(f"Memory used: {model.get_memory_footprint()/1e9:.2f} GB")
model.print_trainable_parameters()

# Loading validation set

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

dataset = load_dataset("spider")
validation_df = pd.DataFrame(dataset['validation'])
train_df = pd.DataFrame(dataset['train'])
validation_df.head(3)

# Loading schema dataset

In [ ]:
spider_tables = load_dataset("richardr1126/spider-schema", split="train")
df= pd.DataFrame(spider_tables)

print(spider_tables)

In [ ]:
schema_lookup ={}
for row in spider_tables:
  schema_lookup[row['db_id']]= row

schema_lookup['csu_1']

## Understanding table Counts

In [ ]:
def total_tables(schema):
    count = 0
    count = sum(1 for char in schema if char =='|')
    return count+1

# total_tables = total_tables('Campuses : Id (number) , Campus (text) , Location (text) , County (text) , Year (number) | csu_fees : Campus (number) , Year (number) , CampusFee (number) | degrees : Year (number) , Campus (number) , Degrees (number) | discipline_enrollments : Campus (number) , Discipline (number) , Year (number) , Undergraduate (number) , Graduate (number) | enrollments : Campus (number) , Year (number) , TotalEnrollment_AY (number) , FTE_AY (number) | faculty : Campus (number) , Year (number) , Faculty (number)')
# total_tables
    
df['count_tables'] = df['Schema (values (type))'].apply(total_tables)
count_tables = df['count_tables'].tolist()

import matplotlib.pyplot as plt

# Create the histogram
plt.hist(count_tables, bins=10, edgecolor='black')

# Add labels and title
plt.title('Table_count_distribution_per_database')
plt.xlabel('Table_count')
plt.ylabel('Frequency')

# Display the plot
plt.show()
print(f'mean count of tables is : {np.mean(count_tables)}')

    

In [ ]:

train_df['schema'] = train_df['db_id'].apply(lambda db_id: schema_lookup[db_id]['Schema (values (type))'] )

train_df['count_tables'] = train_df['schema'].apply(total_tables)
count_tables = train_df['count_tables'].tolist()

import matplotlib.pyplot as plt

# Create the histogram
plt.hist(count_tables, bins=10, edgecolor='black')

# Add labels and title
plt.title('Table_count_distribution_per_example in Training set')
plt.xlabel('Table_count')
plt.ylabel('Frequency')

# Display the plot
plt.show()
print(f'mean count of tables in training_set : {np.mean(count_tables)}')


In [ ]:

validation_df['schema'] =       validation_df['db_id'].apply(lambda db_id: schema_lookup[db_id]['Schema (values (type))'] )
validation_df['count_tables'] = validation_df['schema'].apply(total_tables)
count_tables = validation_df['count_tables'].tolist()

import matplotlib.pyplot as plt

# Create the histogram
plt.hist(count_tables, bins=10, edgecolor='black')

# Add labels and title
plt.title('Table_count_distribution_per_example in validation set')
plt.xlabel('Table_count')
plt.ylabel('Frequency')

# Display the plot
plt.show()
print(f'mean count of tables in validation_set : {np.mean(count_tables)}')


# Creating prompt = Instruct: ....... Schema: ........... Question: 

In [ ]:
def parse_primary_keys(pk_string):

  pk_map ={}
  # There are database full of tables but has not primary keys at all.
  if not pk_string:
    return pk_map

  for entry in pk_string.split('|'):
    entry = entry.strip()
    table, cols = entry.split(':')

    pk_map[table.strip()] =cols.strip()
  return pk_map

db = 'csu_1'
map = parse_primary_keys(pk_string = schema_lookup[db]['Primary Keys'])
print(map)

In [ ]:
def parse_foreign_keys(fk_string):
    fk_map = {}
    words = fk_string
    fk_string = words.split()
    for i,word in enumerate(fk_string):
        if word =='equals':
            first_table = fk_string[i-3]
            second_table = fk_string[i+1]

            fk_map[(first_table, fk_string[i-1])] = (second_table, fk_string[i+3])


    return fk_map
map = parse_foreign_keys(fk_string = schema_lookup[db]['Foreign Keys'])
print(map)

In [ ]:
def format_schema(db_id):
    row = schema_lookup[db_id]
    schema = row['Schema (values (type))']
    pk_map = parse_primary_keys(row['Primary Keys'])
    fk_map = parse_foreign_keys(row['Foreign Keys'])

    schema_lines = []
    for table_info in schema.split('|'):

        table_info = table_info.strip()

        table_name, cols_info = table_info.split(':')
        table_name = table_name.strip()
        cols_info = cols_info.strip()

        # a table can have no primary key.
        pk =''
        if table_name in pk_map:
            pk = pk_map[table_name]

        col_lines = []
        for col_info in cols_info.split(','):

            col_info = col_info.strip()

            col_name = col_info[:col_info.rfind('(')].strip()
            col_type = col_info[col_info.rfind('(')+1 : col_info.rfind(')')].strip()

            # checking if foreign key exist for this column
            fk_found = False
            col_to_find = col_name


            if (table_name, col_to_find) in fk_map:
                mapping = fk_map[(table_name, col_to_find)]
                fk_found = True


            annotations = []

            annotations.append(f'{ col_type}')
            if pk and col_name == pk:
                annotations.append(' PK')
            if fk_found:
                temp = f'{mapping[0]}.{mapping[1]}'
                annotations.append(f' FK -> {temp}')

            annotations = f'- {col_name} ' +  f'({','.join(annotations)})'

            col_lines.append(annotations)
        col_lines = '\n'.join(col_lines)
        schema_lines.append(f'# Table: {table_name} \n## Columns:\n{col_lines}')
    s = '\n\n'.join(schema_lines)

    schema_lookup[db_id]['format_schema'] = s
    # print (f' schema is{schema_lookup[db_id]['format_schema']}')
    
    return s

db_id = 'concert_singer'
s = format_schema(db_id)

print(s)

In [ ]:
schema_table= pd.DataFrame(spider_tables)
schema_table['format_schema'] = df['db_id'].apply(format_schema)

In [ ]:
shortest_schema = min(schema_table['format_schema'],key= len)
approx_tokens = shortest_schema.split()
len(approx_tokens)

In [ ]:
# # TABLE LEVEL SIMILARITY
# import re

# def ques_table_similarity(schema, question):
    
#     table_schemas = schema.split('\n\n')

#     sim_dic ={}

#     for table_schema in table_schemas:
    
#         match1 = re.search(r':(.*)\n', table_schema) # Captures  (everything between : and \n)
#         table_name = match1.group(1).strip()

#         table_embeddings = sim_model.encode(table_name)
#         ques_embeddings =   sim_model.encode(question)
#         cs = util.cos_sim(table_embeddings, ques_embeddings)

#         sim_dic[table_name] = cs
#     return sim_dic

# question = train_df.iloc[4499]['question']
# sim_dic = ques_table_similarity( s, question)

# print(f'question: {question}')
# # print(s)
# sim_dic

In [ ]:
# # TABLE+ COLUMNS+ SCHEMA MAP LEVEL SIMILARITY

# def ques_SCHEMA_similarity( schema, question):
    
#     table_schemas = schema.split('\n\n')

#     sim_dic ={}

#     for table_schema in table_schemas:
    
#         match1 = re.search(r':(.*)\n', table_schema) # Captures  (everything between : and \n)
#         table_name = match1.group(1).strip()

#         schema_embeddings = sim_model.encode(table_schema)
#         ques_embeddings =   sim_model.encode(question)
#         cs = util.cos_sim(schema_embeddings, ques_embeddings)

#         sim_dic[table_name] = cs
#     return sim_dic

# question = validation_df.iloc[4]['question']
# sim_dic = ques_SCHEMA_similarity( s, question)

# print(f'question: {question}')
# # print(s)
# sim_dic
        

In [ ]:
# def schema_pruning(schema_lookup, index, dataset):  

#     db_id = dataset.iloc[index]['db_id']
#     question = dataset.iloc[index['question']]
#     schema = format_schema(db_id)

#     dic ={}
#     print('dic created/')

#     table_schemas = schema.split('\n\n')

    
#     for table_schema in table_schemas:
        
#         match1 = re.search(r':(.*)\n', table_schema) # Captures  (everything between : and \n)
#         table_name = match1.group(1).strip()
#         # print(table_name)

#         match2 = re.search(rf"{'\n##'}\s+(.*)", table_schema, re.DOTALL)
#         value = match2.group(1)
#         # print(value)

#         dic[table_name] = value

#     return dic

# dic = schema_pruning(schema_lookup, 0, dataset = validation_df)
# dic

# GENERATION

In [ ]:
def build_prompt( schema_lookup, index, dataset, inference = False ):

    df = dataset
    prompt = ''

    db_id = df['db_id'][index]
    question = df['question'][index]
    schema = schema_lookup[db_id]['format_schema']
    
    prompt = f"""<|start_header_id|>system<|end_header_id|>

Convert the natural language question to SQL using the schema below. Use SQLite syntax and SQLite functions only.{tokenizer.eos_token}<|start_header_id|>user<|end_header_id|>

### Schema:
{schema}

### Question:
{question + tokenizer.eos_token}"""

    if inference == False:
        query = df['query'][index]
        text = prompt+ f"""<|start_header_id|>assistant<|end_header_id|>

{query + tokenizer.eos_token}"""
        
        return {'text':text}
    return{'text': prompt}

# train_df = pd.DataFrame(dataset['train'])
# validation_df = pd.DataFrame(dataset['validation'])

text = build_prompt( schema_lookup, index =0,dataset = validation_df, inference = True )
print(text)
    

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenized = tokenizer(['i love you',' do i love you'],padding = True,padding_side = 'left', )
tokenized

In [ ]:
tokenizer.decode(128000)

In [ ]:
# inference_texts = []

# for i in range(len(validation_df)):
#     text = build_prompt( schema_lookup, index =i ,dataset = validation_df, inference = True )
#     text = text['text']
#     inference_texts.append(text)

# tokenized_examples = tokenizer(inference_texts,
#                               padding = 'left',
#                               )



In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Step 1 — build prompts WITHOUT the answer (inference=True)
inference_texts = []
for i in range(len(validation_df)):
    result = build_prompt(schema_lookup, i, validation_df, inference=True)
    inference_texts.append(result['text'])

# Step 2 — tokenize all prompts
tokenized = tokenizer(
    inference_texts,
    truncation=True,
    max_length=2048,
    padding=False,          # don't pad yet — DataLoader will handle per batch
    add_special_tokens= True,
    return_tensors=None,    # return list, not tensors
)

# Step 3 — build a simple dataset
from datasets import Dataset
inference_dataset = Dataset.from_dict({
    'input_ids': tokenized['input_ids'],
    'attention_mask': tokenized['attention_mask'],
})

# Step 4 — use DataCollatorWithPadding to batch and pad
collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors='pt',
)

dataloader = DataLoader(
    inference_dataset,
    batch_size=16,
    shuffle=False,           # keep order so predictions align with val_df rows
    collate_fn=collator,
)

In [ ]:
import warnings
import transformers
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

In [ ]:
import time
torch.cuda.reset_peak_memory_stats()

start = time.time()
batch = next(iter(dataloader))
input_ids = batch['input_ids'].to('cuda')
attention_mask = batch['attention_mask'].to('cuda')

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        pad_token_id=tokenizer.pad_token_id,
    )

elapsed = time.time() - start
peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"Time: {elapsed:.1f}s, Peak memory: {peak:.2f}GB")

In [ ]:
# from unsloth import FastLanguageModel

# FastLanguageModel.for_inference(model)  # unsloth inference mode — 2x faster

from tqdm.auto import tqdm
predictions = []

model.eval()
for batch in tqdm(dataloader, desc = 'Generating'):
    input_ids      = batch['input_ids'].to('cuda')
    attention_mask = batch['attention_mask'].to('cuda')

    # GREEDY SEARCH
    with torch.no_grad():
        outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False,        # greedy
        use_cache=True,         # works fine with greedy
        eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        pad_token_id=tokenizer.pad_token_id,
    )

        ## BEAM SEARCH 
        # outputs = model.generate(
        #     input_ids=input_ids,
        #     attention_mask=attention_mask,
        #     max_new_tokens=128,
        #     do_sample=False,
        #     num_beams=4,                    # beam search with 4 beams
        #     early_stopping=True,            # stop when all beams hit eos
        #     no_repeat_ngram_size=3,         # prevents repetitive SQL clauses
        #     use_cache= True, 
        #     eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        #     pad_token_id=tokenizer.pad_token_id,
        # )
    # Slice off the prompt — outputs include input tokens too
    # for i, output in enumerate(outputs):
    #     prompt_len = input_ids.shape[1]  # works because all prompts padded to same length
    #     generated = output[prompt_len:]
    #     sql = tokenizer.decode(generated, skip_special_tokens=True).strip()
    #     predictions.append(sql)

    for i, output in enumerate(outputs):
        # print(output)
        actual_prompt_len = batch['attention_mask'][i].sum().item()
        generated = output[batch['input_ids'].shape[1]:]
        sql = tokenizer.decode(generated, skip_special_tokens=True).strip()
        if sql.startswith("assistant"):
            sql = sql[len("assistant"):].strip()
        predictions.append(sql)

print(predictions[:3])

In [ ]:
import json
with open("/kaggle/working/predictions.json", "w") as f:
    json.dump(predictions, f)

In [ ]:
# Save predictions
import pandas as pd
validation_df['predicted_sql'] = predictions
validation_df.to_csv("/kaggle/working/validation_predictions.csv", index=False)

# Save model adapter too if not already saved
model.save_pretrained("/kaggle/working/final_adapter")
tokenizer.save_pretrained("/kaggle/working/final_adapter")

In [ ]:
# def generate_sql( index, model, tokenizer, max_new_tokens=256):
#     # Build inference prompt (no SQL appended)
    
#     question = validation_df['question'][index]
#     db_id    = validation_df['db_id'][index]
    
#     prompt = build_prompt( schema_lookup, index ,dataset = validation_df, inference = True )
#     prompt = prompt['text']

#     # Tokenize
#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=2048,
#         padding_side = 'left'
#     ).to(model.device)
    
#     # print(f'input_token_ids:{inputs['input_ids'][0]}')
#     # print(f'input_token_ids_shape:{inputs['input_ids'][0].shape}')

#     # Generate
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=False,        # don't want the model to be "creative" or "poetic" with your database queries; you want the most mathematically probable token every time
#             temperature=1,
#             num_beams = 4,
#             early_stopping = True
#         )
#     # print(f'output_token_ids:{outputs[0]}')
#     # print(f'output_token_ids_shape:{outputs[0].shape}')

#     # Decode only the generated tokens (not the prompt)
#     generated = outputs[0][inputs['input_ids'].shape[1]:]
#     sql = tokenizer.decode(generated, skip_special_tokens=True).strip()
#     # Only take the first line to ignore the "Tutorial Mode" repetitions
#     # final_sql = sql.split("\n")[0].split(";")[0].strip()
#     # return final_sql,sql
#     return sql


# # Test on one example
# index = 2
# question = validation_df['question'][index]
# db_id    = validation_df['db_id'][index]
# sql = generate_sql(question, db_id, model, tokenizer)
# print(f"Question     : {question}")
# # print(f"Prompt     : {prompt}")
# print(f"Predicted SQL: {sql}")
# print(f"Ground truth : {validation_df['query'][index]}")


In [ ]:
schema_table= pd.DataFrame(spider_tables)
schema_table['format_schema'] = df['db_id'].apply(format_schema)

In [63]:
import sqlite3

KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
def execute_sql(sql, db_id, db_dir = KAGGLE_DB_DIR ):
    
    """Execute SQL against SQLite DB, return result set or None on error."""
    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

    # CRITICAL: Check if the file actually exists before connecting
    if not os.path.exists(db_path):
        print(f"Warning: Database file not found at {db_path}")
        return None
    try:
        # 'mode=ro' tells SQLite to open it in Read-Only mode
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        cursor = conn.cursor()
        cursor.execute(sql)
        result = cursor.fetchall()
        conn.close()
        return result
    except Exception as e:
        return None   # invalid SQL

sql = 'SELECT name ,  country ,  age FROM singer ORDER BY age DESC'
result = execute_sql(sql, db_id = 'concert_singer')
print(result)

[('Joe Sharp', 'Netherlands', 52), ('John Nizinik', 'France', 43), ('Rose White', 'France', 41), ('Timbaland', 'United States', 32), ('Justin Brown', 'France', 29), ('Tribal King', 'France', 25)]


In [67]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
import threading

# SQLite connections are not thread-safe, so create one per thread
thread_local = threading.local()

def get_connection(db_path):
    if not hasattr(thread_local, 'connections'):
        thread_local.connections = {}
    if db_path not in thread_local.connections:
        thread_local.connections[db_path] = sqlite3.connect(db_path)
    return thread_local.connections[db_path]

def evaluate_single(args):
    i, predicted, ground_truth, db_id = args
    pred_result  = execute_sql(predicted, db_id)
    truth_result = execute_sql(ground_truth, db_id)
    
    if pred_result is None:
        return i, 'error'
    elif Counter(pred_result) == Counter(truth_result):
        return i, 'correct'
    else:
        return i, 'wrong'

def execution_accuracy_parallel(validation_df, predictions, max_workers=4):
    total   = len(validation_df)
    correct = 0
    errors  = 0

    args_list = [
        (
            i,
            predictions[i],
            validation_df['query'].iloc[i],
            validation_df['db_id'].iloc[i],
        )
        for i in range(total)
    ]

    results = [None] * total

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(evaluate_single, args): args[0] for args in args_list}
        for future in tqdm(as_completed(futures), total=total, desc="Evaluating"):
            i, status = future.result()
            results[i] = status

    correct = sum(1 for r in results if r == 'correct')
    errors  = sum(1 for r in results if r == 'error')
    accuracy = correct / total

    print(f"Execution Accuracy : {accuracy:.4f} ({correct}/{total})")
    print(f"Invalid SQL errors : {errors}/{total}")
    return accuracy

In [69]:
# Step 2 — evaluation (CPU, parallelized, no GPU needed)
accuracy = execution_accuracy_parallel(validation_df, predictions, max_workers=4)


Evaluating:   0%|          | 0/1034 [00:00<?, ?it/s]

Execution Accuracy : 0.7002 (724/1034)
Invalid SQL errors : 56/1034


In [65]:
# from collections import Counter
# def execution_accuracy(val_df, model, tokenizer):
#     correct = 0
#     total   = len(val_df)
#     errors  = 0

#     for i in range(total):
#         question     = val_df['question'].iloc[i]
#         db_id        = val_df['db_id'].iloc[i]
#         ground_truth = val_df['query'].iloc[i]

#         # Generate predicted SQL
#         predicted = predictions[i]

#         # Execute both
#         pred_result  = execute_sql(predicted,    db_id)
#         # pred_result =  execute_sql('SELECT count(*) FROM singer', db_id)
#         truth_result = execute_sql(ground_truth, db_id)

#         # print(f'pred_result : {pred_result}')
#         # print(f'truth_result : {truth_result}')
        
#         # Compare results
#         if pred_result is None:
#             errors += 1
#         elif Counter(pred_result) == Counter(truth_result):
#             correct += 1
            

#         print(f'correct_% at i= { i} : {correct/(i+1)}')

#     accuracy = correct / total
#     print(f"Execution Accuracy : {accuracy:.4f} ({correct}/{total})")
#     print(f"Invalid SQL errors : {errors}/{total}")
#     return accuracy

In [ ]:
# acc = execution_accuracy(validation_df, model, tokenizer)
# print(acc)

In [ ]:
# # Test on one example
# index = 32

# question = validation_df['question'][index]
# db_id    = validation_df['db_id'][index]
# sql = generate_sql(question, db_id, model, tokenizer)
# print(f"Question     : {question}")
# # print(f"Prompt     : {prompt}")
# print(f"Predicted SQL: {sql}")
# print(f"Ground truth : {validation_df['query'][index]}")

In [ ]:
print(s)